In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
import librosa.display
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
from mutagen import File
import soundfile as sf

In [4]:
import re
from collections import defaultdict

In [5]:
import shutil

In [6]:
audio_folder = 'C:/Users/LShel/Downloads/A (1)/A -'
base_output_path = "C:/Users/LShel/OneDrive/Documents/Applied_Machine_Learning/Datasets/Data"

In [7]:
key_patterns = {
    'Cm': [r'C\s*(min|minor)'],
    'C#m': [r'C#\s*(min|minor)', r'Db\s*(min|minor)'],
    'Dm': [r'D\s*(min|minor)'],
    'D#m': [r'D#\s*(min|minor)', r'Eb\s*(min|minor)'],
    'Em': [r'E\s*(min|minor)'],
    'Fm': [r'F\s*(min|minor)'],
    'F#m': [r'F#\s*(min|minor)', r'Gb\s*(min|minor)'],
    'Gm': [r'G\s*(min|minor)'],
    'G#m': [r'G#\s*(min|minor)', r'Ab\s*(min|minor)', r'G#m'],
    'Am': [r'A\s*(min|minor)'],
    'A#m': [r'A#\s*(min|minor)', r'Bb\s*(min|minor)', r'A#m'],
    'Bm': [r'B\s*(min|minor)'],
}

In [8]:
sorted_audio = defaultdict(list)

for file in os.listdir(audio_folder):
    if file.endswith(('.wav', '.mp3', '.ogg')):
        found = False
        for key, patterns in key_patterns.items():
            for pattern in patterns:
                if re.search(pattern, file, re.IGNORECASE):
                    sorted_audio[key].append(os.path.join(audio_folder, file))
                    found = True
                    break
            if found:
                break
        if not found:
            sorted_audio['Unknown'].append(os.path.join(audio_folder, file))

for key, files in sorted_audio.items():
    output_path = os.path.join(base_output_path, key)
    os.makedirs(output_path, exist_ok=True)
        
    for file_path in files:
        file_name = os.path.basename(file_path)
        dest_path = os.path.join(output_path, file_name)
        
        try:
            shutil.copy2(file_path, dest_path)
        except Exception as e:
            print(f"    Error copying {file_name}: {e}")

In [10]:
# ----- #

In [9]:
root_folder = 'C:/Users/LShel/OneDrive/Documents/Applied_Machine_Learning/Datasets/Data'

keys = ['Cm', 'C#m', 'Dm', 'D#m', 'Em', 'Fm', 'F#m', 'Gm', 'G#m', 'Am', 'A#m', 'Bm']

audio_files = []
for k in keys:
    key_folder_path = f"{root_folder}/{k}"
    for file in os.listdir(key_folder_path):
        if file.endswith(('.wav', '.mp3', '.ogg')):
            audio_files.append(f"{key_folder_path}/{file}")

durations = []
for file_path in audio_files:
    audio = File(file_path)
    if audio is not None and audio.info is not None:
        durations.append(audio.info.length * 1000) 

min_duration = min(durations)
print(f"Global shortest duration: {min_duration:.2f} ms")

Global shortest duration: 6164.88 ms


In [11]:
for k in keys:

    key_files = []
    
    key_folder_path = f"{root_folder}/{k}"
    
    for file in os.listdir(key_folder_path):
        if file.endswith(('.wav', '.mp3', '.ogg')):
            key_files.append(f"{key_folder_path}/{file}")

    key_output_path = f"{root_folder}/{k}_t"
    os.makedirs(key_output_path, exist_ok=True)

    name_inc = 0

    for file in key_files:

        save_path = f"{key_output_path}/{name_inc}.mp3"
        name_inc = name_inc + 1
        
        data, samplerate = sf.read(file)
        target_samples = int((min_duration / 1000) * samplerate)

        trimmed_data = data[:target_samples]

        # filename = os.path.basename(file)

        sf.write(save_path, trimmed_data, samplerate)

In [12]:
def create_spectrogram(audio_file, image_file):
    fig = plt.figure()
    ax = fig.add_subplot(1, 1, 1)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

    y, sr = librosa.load(audio_file)
    ms = librosa.feature.melspectrogram(y=y, sr=sr)
    log_ms = librosa.power_to_db(ms, ref=np.max)
    librosa.display.specshow(log_ms, sr=sr)

    fig.savefig(image_file)
    plt.close(fig)
    
def create_pngs_from_mp3s(input_path, output_path):
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    dir = os.listdir(input_path)

    for i, file in enumerate(dir):
        input_file = os.path.join(input_path, file)
        output_file = os.path.join(output_path, file.replace('.mp3', '.png'))
        create_spectrogram(input_file, output_file)

In [13]:
for k in keys:
    input_path = f"C:/Users/LShel/OneDrive/Documents/Applied_Machine_Learning/Datasets/Data/{k}_t"
    output_path = f"C:/Users/LShel/OneDrive/Documents/Applied_Machine_Learning/Datasets/Data/Spectrogram_Data/{k}"
    create_pngs_from_mp3s(input_path, output_path)

C:\Users\LShel\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [14]:
from keras.preprocessing import image

def load_images_from_path(path, label):
    images = []
    labels = []

    for file in os.listdir(path):
        images.append(image.img_to_array(image.load_img(os.path.join(path, file), target_size=(224, 224, 3))))
        labels.append((label))
        
    return images, labels

def show_images(images):
    fig, axes = plt.subplots(1, 8, figsize=(20, 20), subplot_kw={'xticks': [], 'yticks': []})

    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i] / 255)
        
x = []
y = []

In [15]:
label_counter = 0

for k in keys:
    
    images, labels = load_images_from_path(f"C:/Users/LShel/OneDrive/Documents/Applied_Machine_Learning/Datasets/Data/Spectrogram_Data/{k}", label_counter)
    
    x += images
    y += labels

    label_counter = label_counter + 1

In [16]:
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y, test_size=0.2, random_state=0)

x_train_norm = np.array(x_train) / 255
x_test_norm = np.array(x_test) / 255

y_train_encoded = to_categorical(y_train)
y_test_encoded = to_categorical(y_test)

In [17]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Flatten, Dense

model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Flatten())
model.add(Dense(1024, activation='relu'))
model.add(Dense(12, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\LShel\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 128)  │        36,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │    18,875,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │        12,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,220,748 (73.32 MB)

 Trainable params: 19,220,748 (73.32 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
hist = model.fit(x_train_norm, y_train_encoded, validation_data=(x_test_norm, y_test_encoded), batch_size=10, epochs=20)

Epoch 1/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 11s 443ms/step - accuracy: 0.1308 - loss: 2.5563 - val_accuracy: 0.1333 - val_loss: 2.4719
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 8s 418ms/step - accuracy: 0.0956 - loss: 2.4727 - val_accuracy: 0.1333 - val_loss: 2.4799
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 7s 414ms/step - accuracy: 0.1374 - loss: 2.4714 - val_accuracy: 0.1333 - val_loss: 2.4618
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 11s 424ms/step - accuracy: 0.1918 - loss: 2.3974 - val_accuracy: 0.1778 - val_loss: 2.4746
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 8s 423ms/step - accuracy: 0.0953 - loss: 2.4755 - val_accuracy: 0.1111 - val_loss: 2.4712
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 8s 415ms/step - accuracy: 0.1590 - loss: 2.4485 - val_accuracy: 0.1778 - val_loss: 2.5053
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 8s 426ms/step - accuracy: 0.1450 - loss: 2.4252 - val_accuracy: 0.1778 - val_loss: 2.5440
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 8s 418ms/step - accuracy: 0.1529 - loss: 2.3250 - val_accuracy: 